AQUEST FITXER ACONSEGUEIX BAIXAR LES DADES FUNADMENTALS DELS ANYS 2009 FINS 2014

PASS 1) BAIXA ELS FILINGS

In [26]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup

# -------- CONFIG --------
USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
CIK = "0000320193"   # Apple
FORMS = {"10-K", "10-Q"}
PAUSE = 0.2

# ---- JSON SOURCE ----
def get_filings_json(cik):
    sub_url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    j = requests.get(sub_url, headers=HEADERS, timeout=30).json()

    def pack_from_recent(recent_dict):
        return pd.DataFrame({
            "accession": recent_dict["accessionNumber"],
            "form": recent_dict["form"],
            "filed": recent_dict["filingDate"],
            "primary": recent_dict["primaryDocument"],
            "source": "json"
        })

    recent_df = pack_from_recent(j["filings"]["recent"])
    older_parts = []
    for f in j.get("filings", {}).get("files", []):
        url = f"https://data.sec.gov/submissions/{f['name']}"
        y = requests.get(url, headers=HEADERS, timeout=30).json()
        if "filings" in y and "recent" in y["filings"]:
            older_parts.append(pack_from_recent(y["filings"]["recent"]))
        time.sleep(PAUSE)

    all_filings = pd.concat([recent_df] + older_parts, ignore_index=True)
    all_filings["filed"] = pd.to_datetime(all_filings["filed"])
    return all_filings

# ---- ATOM SOURCE ----
def get_filings_atom(cik, form_type="10-K"):
    url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type={form_type}&owner=exclude&count=100&output=atom"
    r = requests.get(url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(r.text, "xml")
    entries = []
    for entry in soup.find_all("entry"):
        filed = entry.find("filing-date").text if entry.find("filing-date") else None
        link = entry.find("link")["href"] if entry.find("link") else None
        acc = entry.find("accession-number").text if entry.find("accession-number") else None
        entries.append((acc, form_type, filed, link))
    return entries

def get_all_atom(cik):
    filings = []
    for form in FORMS:
        filings += get_filings_atom(cik, form)
        time.sleep(PAUSE)
    df = pd.DataFrame(filings, columns=["accession","form","filed","primary"])
    df["filed"] = pd.to_datetime(df["filed"])
    df["source"] = "atom"
    return df

# ---- FUSION ----
def merge_sources(cik):
    df_json = get_filings_json(cik)
    df_atom = get_all_atom(cik)

    # fusionem, mantenim accession + form + filed
    all_filings = pd.concat([df_json, df_atom], ignore_index=True)

    # treure duplicats
    all_filings = all_filings.drop_duplicates(subset=["accession","form","filed"])

    # filtrem només 10-K / 10-Q i des del 2000
    all_filings = all_filings[all_filings["form"].isin(FORMS)]
    all_filings = all_filings[all_filings["filed"] >= "2000-01-01"]

    return all_filings.sort_values("filed")

def main():
    filings = merge_sources(CIK)
    filings.to_csv("all_filings_2000_apple.csv", index=False)
    print(f"Saved {len(filings)} filings to all_filings_2000_apple.csv")
    print(filings.head(10))
    print("...")
    print(filings.tail(10))

if __name__ == "__main__":
    main()


Saved 105 filings to all_filings_2000_apple.csv
                 accession  form      filed  \
1085  0000912057-00-003201  10-Q 2000-02-01   
1084  0000912057-00-023442  10-Q 2000-05-11   
1083  0000912057-00-033901  10-Q 2000-07-31   
1129  0000912057-00-053623  10-K 2000-12-14   
1082  0000912057-01-004642  10-Q 2001-02-12   
1081  0000912057-01-515409  10-Q 2001-05-14   
1080  0000912057-01-528148  10-Q 2001-08-13   
1128  0000912057-01-544436  10-K 2001-12-21   
1079  0000912057-02-004945  10-Q 2002-02-11   
1078  0000912057-02-020280  10-Q 2002-05-14   

                                                primary source  
1085  https://www.sec.gov/Archives/edgar/data/320193...   atom  
1084  https://www.sec.gov/Archives/edgar/data/320193...   atom  
1083  https://www.sec.gov/Archives/edgar/data/320193...   atom  
1129  https://www.sec.gov/Archives/edgar/data/320193...   atom  
1082  https://www.sec.gov/Archives/edgar/data/320193...   atom  
1081  https://www.sec.gov/Archives/edgar/dat

PASS 2) BAIXA FUNDAMENTALS DEL 2009 FINS AL 2014

In [27]:
# ---------- financials_2009_2014.py ----------
import re, io, zipfile, time, requests, pandas as pd
from urllib.parse import urljoin
from lxml import etree
from datetime import datetime

USER_AGENT = "Jon jonmcgowan15@gmail.com"
HEADERS = {"User-Agent": USER_AGENT}
PAUSE = 0.2  # pausa lleu entre filings per ser amables amb la SEC

DATE_FROM = "2009-07-22"
DATE_TO   = "2014-07-23"

USGAAP_TAGS = {
    "revenue": [
        "SalesRevenueNet",
        "RevenueFromContractWithCustomerExcludingAssessedTax",
        "Revenues"
    ],
    "net_income": ["NetIncomeLoss"],
    "assets": ["Assets"],
    "liabilities": ["Liabilities"],
    "equity": [
        "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
        "StockholdersEquity"
    ]
}

# ------------------- HTTP helpers with retry -------------------
def _get(url, timeout=60, tries=4, backoff=0.9):
    """
    GET amb reintents per gestionar 403/429/temps i errors intermitents.
    backoff: temps d'espera incremental (s).
    """
    last_err = None
    for k in range(tries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            if r.status_code in (403, 429):
                time.sleep(backoff * (k + 1))
                continue
            r.raise_for_status()
            return r
        except Exception as e:
            last_err = e
            time.sleep(backoff * (k + 1))
    raise last_err

def fetch_json(url, timeout=60):
    return _get(url, timeout=timeout).json()

def fetch_bytes(url, timeout=90):
    return _get(url, timeout=timeout).content

# ------------------- Paths & discovery -------------------
def derive_base_dir(index_url: str) -> str:
    """
    Retorna .../Archives/edgar/data/{cik}/{accession_sense_guions}/
    a partir d'un -index.htm(l) d'EDGAR (ATOM).
    """
    m = re.search(r"/data/(\d+)/(\d{10}-\d{2}-\d{6})(?:/|)", index_url)
    if m:
        cik = m.group(1)
        acc_no_dash = m.group(2).replace("-", "")
        return f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc_no_dash}/"
    # Fallback per segments
    parts = index_url.split("/")
    for i, seg in enumerate(parts):
        if re.fullmatch(r"\d{10}-\d{2}-\d{6}", seg or "") and i >= 2:
            cik = parts[i - 1]
            return f"https://www.sec.gov/Archives/edgar/data/{cik}/{seg.replace('-', '')}/"
    # Últim recurs
    return index_url.rsplit("/", 1)[0] + "/"

def candidates_from_indexjson(base_dir: str):
    """
    Retorna candidats plausibles d’instància XBRL (urls completes) des de index.json,
    prioritzant .xml/.xbrl/.ins, després .zip, i finalment (per si de cas) .htm/.html (inline).
    """
    exts_order = [".xml", ".xbrl", ".ins", ".zip", ".html", ".htm"]
    meta = fetch_json(base_dir + "index.json")
    items = meta.get("directory", {}).get("item", [])
    cands = []
    for it in items:
        name = it.get("name", "")
        low = name.lower()
        if any(low.endswith(ext) for ext in exts_order):
            # descarta esquemes/linkbases quan no són l’instància
            if any(sfx in low for sfx in ["_cal", "_def", "_lab", "_pre", ".xsd"]):
                continue
            cands.append((exts_order.index(next(ext for ext in exts_order if low.endswith(ext))), it.get("size", 0), base_dir + name))
    # ordena per prioritat d’extensió i per mida descendent
    cands.sort(key=lambda t: (t[0], -t[1]))
    return [u for _,__,u in cands]

def candidates_from_index_page(index_url: str):
    """
    Extracte simple d’enllaços des de la pàgina Documents (fallback).
    """
    html = _get(index_url, timeout=60).text
    hrefs = re.findall(r'href="([^"]+)"', html, flags=re.I)
    hrefs = [urljoin(index_url, h) for h in hrefs]
    exts_order = [".xml", ".xbrl", ".ins", ".zip", ".html", ".htm"]
    cands = []
    for u in hrefs:
        low = u.lower()
        if any(low.endswith(ext) for ext in exts_order):
            if any(sfx in low for sfx in ["_cal", "_def", "_lab", "_pre", ".xsd"]):
                continue
            cands.append((exts_order.index(next(ext for ext in exts_order if low.endswith(ext))), u))
    cands.sort(key=lambda t: t[0])
    return [u for _,u in cands]

def sniff_instance_from_bytes(blob: bytes) -> etree._Element | None:
    """
    Intenta parsejar bytes com a instància XBRL i retorna l'arrel si té contexts.
    """
    try:
        parser = etree.XMLParser(recover=True, huge_tree=True)
        root = etree.fromstring(blob, parser=parser)
    except Exception:
        return None
    # Ha de contenir contexts
    if root.find(".//{*}context") is None:
        return None
    return root

def find_xbrl_instance_and_root(index_url: str):
    """
    Prova candidats (index.json -> pàgina) i detecta l’instància real.
    Suporta ZIP: obre i prova els .xml interns.
    Retorna (label_per_log, root_element)
    """
    base = derive_base_dir(index_url)
    tried = []

    def candidate_streams():
        # 1) index.json
        try:
            for u in candidates_from_indexjson(base):
                yield u
        except Exception:
            pass
        # 2) index page
        try:
            for u in candidates_from_index_page(index_url):
                yield u
        except Exception:
            pass

    for url in candidate_streams():
        low = url.lower()
        tried.append(url)
        try:
            if low.endswith(".zip"):
                blob = fetch_bytes(url)
                with zipfile.ZipFile(io.BytesIO(blob)) as zf:
                    # prova xml/xbrl/ins interns, prioritzant mida
                    names = [(zi.filename, zi.file_size) for zi in zf.infolist()
                             if re.search(r"\.(xml|xbrl|ins)$", zi.filename, re.I)
                             and not re.search(r"(_cal|_def|_lab|_pre|\.xsd)$", zi.filename, re.I)]
                    names.sort(key=lambda t: -t[1])
                    for name,_ in names:
                        root = sniff_instance_from_bytes(zf.read(name))
                        if root is not None:
                            return (f"{url}::{name}", root)
            else:
                blob = fetch_bytes(url)
                root = sniff_instance_from_bytes(blob)
                if root is not None:
                    return (url, root)
        except Exception:
            continue

    raise RuntimeError("No usable XBRL instance found. Tried: " + ", ".join(tried[:6]) + (" ..." if len(tried) > 6 else ""))

# ------------------- XBRL parsing -------------------
def parse_xbrl_root(root: etree._Element) -> dict | None:
    """
    Extreu mètriques de l’arrel XBRL triant, per a cada tag,
    el context amb endDate (o instant) més recent.
    """
    ns = root.nsmap
    contexts: dict[str, tuple[str, datetime]] = {}

    for ctx in root.findall(".//{*}context", namespaces=ns):
        cid = ctx.get("id")
        if not cid:
            continue
        inst = ctx.find(".//{*}instant", namespaces=ns)
        start = ctx.find(".//{*}startDate", namespaces=ns)
        end   = ctx.find(".//{*}endDate", namespaces=ns)
        if inst is not None and (inst.text or "").strip():
            try:
                contexts[cid] = ("instant", datetime.strptime(inst.text[:10], "%Y-%m-%d"))
            except:
                pass
        elif start is not None and end is not None and (start.text and end.text):
            try:
                contexts[cid] = ("duration", datetime.strptime(end.text[:10], "%Y-%m-%d"))
            except:
                pass

    def best_fact(local: str, want: str) -> float | None:
        elems = root.findall(f".//{{*}}{local}", namespaces=ns)
        best_dt = datetime(1900, 1, 1)
        best_val = None
        for e in elems:
            cref = e.get("contextRef") or e.get("contextref")
            if not cref or cref not in contexts:
                continue
            ctype, dt = contexts[cref]
            if want == "instant" and ctype != "instant":
                continue
            if want == "duration" and ctype != "duration":
                continue
            txt = (e.text or "").strip()
            if not txt:
                continue
            neg = txt.startswith("(") and txt.endswith(")")
            num = re.sub(r"[^\d\.\-]", "", txt)
            if num in {"", "-", ".", "-.", ".-"}:
                continue
            try:
                val = float(num)
                if neg:
                    val = -val
            except:
                continue
            if dt > best_dt:
                best_dt, best_val = dt, val
        return best_val

    out = {"revenue": None, "net_income": None, "assets": None, "liabilities": None, "equity": None}

    # revenue
    for tag in USGAAP_TAGS["revenue"]:
        v = best_fact(tag, "duration")
        if v is not None:
            out["revenue"] = v
            break
    out["net_income"]  = best_fact("NetIncomeLoss", "duration")
    out["assets"]      = best_fact("Assets", "instant")
    out["liabilities"] = best_fact("Liabilities", "instant")
    for tag in USGAAP_TAGS["equity"]:
        v = best_fact(tag, "instant")
        if v is not None:
            out["equity"] = v
            break

    return out if any(v is not None for v in out.values()) else None

# ------------------- MAIN -------------------
def main(csv_in="all_filings_2000_apple.csv", csv_out="financials_2009_2014.csv"):
    filings = pd.read_csv(csv_in)
    filings["filed"] = pd.to_datetime(filings["filed"])
    subset = filings[(filings["filed"] >= DATE_FROM) & (filings["filed"] <= DATE_TO)].copy().sort_values("filed")

    rows = []
    for _, r in subset.iterrows():
        index_url = str(r["primary"])
        if not index_url.startswith("http"):
            index_url = urljoin("https://www.sec.gov/Archives/", index_url)

        try:
            label, root = find_xbrl_instance_and_root(index_url)  # <-- detecta i retorna root
            metrics = parse_xbrl_root(root)
            if not metrics:
                raise RuntimeError("XBRL parsed but no metrics found")

            rows.append({
                "filing_date": r["filed"].strftime("%Y-%m-%d"),
                "form": r["form"],
                "revenue": metrics["revenue"],
                "net_income": metrics["net_income"],
                "total_assets": metrics["assets"],
                "total_liabilities": metrics["liabilities"],
                "shareholders_equity": metrics["equity"],
                "xbrl_source": label
            })
            print(f"[OK] {r['filed'].date()} {r['form']}  -> {label.split('/')[-1]}")
        except Exception as e:
            print(f"[ERR] {r['filed'].date()} {r['form']}: {e}")
            rows.append({
                "filing_date": r["filed"].strftime("%Y-%m-%d"),
                "form": r["form"],
                "revenue": None, "net_income": None,
                "total_assets": None, "total_liabilities": None, "shareholders_equity": None,
                "xbrl_source": None
            })
        time.sleep(PAUSE)

    out = pd.DataFrame(rows).sort_values(["filing_date","form"])
    out.to_csv(csv_out, index=False)
    print(f"[DONE] Saved {len(out)} rows to {csv_out}")

if __name__ == "__main__":
    main()


[OK] 2009-07-22 10-Q  -> aapl-20090627.xml
[OK] 2009-10-27 10-K  -> aapl-20090926.xml
[OK] 2010-01-25 10-Q  -> aapl-20091226.xml
[OK] 2010-01-25 10-K  -> aapl-20090926.xml
[OK] 2010-04-21 10-Q  -> aapl-20100327.xml
[OK] 2010-07-21 10-Q  -> aapl-20100626.xml
[OK] 2010-10-27 10-K  -> aapl-20100925.xml
[OK] 2011-01-19 10-Q  -> aapl-20101225.xml
[OK] 2011-04-21 10-Q  -> aapl-20110326.xml
[OK] 2011-07-20 10-Q  -> aapl-20110625.xml
[OK] 2011-10-26 10-K  -> aapl-20110924.xml
[OK] 2012-01-25 10-Q  -> aapl-20111231.xml
[OK] 2012-04-25 10-Q  -> aapl-20120331.xml
[OK] 2012-07-25 10-Q  -> aapl-20120630.xml
[OK] 2012-10-31 10-K  -> aapl-20120929.xml
[OK] 2013-01-24 10-Q  -> aapl-20121229.xml
[OK] 2013-04-24 10-Q  -> aapl-20130330.xml
[OK] 2013-07-24 10-Q  -> aapl-20130629.xml
[OK] 2013-10-30 10-K  -> aapl-20130928.xml
[OK] 2014-01-28 10-Q  -> aapl-20131228.xml
[OK] 2014-04-24 10-Q  -> aapl-20140329.xml
[OK] 2014-07-23 10-Q  -> aapl-20140628.xml
[DONE] Saved 22 rows to financials_2009_2014.csv
